In [1]:
from tqdm import tqdm
import os
import pandas as pd
import csv
import json
import numpy as np
import pickle

In [2]:
path_to_baseline_labels = '/path/to/result/outputs.csv'
evaluate_a_compliment = False

In [ ]:
ATTRIBUTE_FILE = '/path/to/CUB_200_2011/attributes/attributes.txt'
LABEL_PATH = '/path/to/CUB_processed/class_attr_data_10/val.pkl'

In [3]:
# from CBM paper
MAPPING = [1, 4, 6, 7, 10, 14, 15, 20, 21, 23, 25, 29, 30, 35, 36, 38, 40, 44, 45, 50, 51, 53, 54, 56, 57, 59, 63, 64, 69, 70, 72, 75, 80, 84, 90, 91, \
    93, 99, 101, 106, 110, 111, 116, 117, 119, 125, 126, 131, 132, 134, 145, 149, 151, 152, 153, 157, 158, 163, 164, 168, 172, 178, 179, 181, \
    183, 187, 188, 193, 194, 196, 198, 202, 203, 208, 209, 211, 212, 213, 218, 220, 221, 225, 235, 236, 238, 239, 240, 242, 243, 244, 249, 253, \
    254, 259, 260, 262, 268, 274, 277, 283, 289, 292, 293, 294, 298, 299, 304, 305, 308, 309, 310, 311]

def get_list_of_used_attributes():
    with open(ATTRIBUTE_FILE, 'r') as f:
        attributes = f.readlines()
    used_attributes = [attributes[m] for m in MAPPING]
    used_attributes = [a.replace('\n', '').split(' ')[1] for a in used_attributes]
    return used_attributes

In [4]:
df = pd.read_csv(path_to_baseline_labels)

In [5]:
used_attributes = get_list_of_used_attributes()
folder_to_attribute_dict = {}
for ua in used_attributes:
    folder_to_attribute_dict[ua.replace('--', '_')] = ua

In [6]:
unique_attribute_types = []
for c in df.columns:
    if c == 'image_path':
        continue
    attr_type = c.split('--')[0]
    if attr_type not in unique_attribute_types:
        unique_attribute_types.append(attr_type)

In [7]:
def get_a_old(bird, changed_attr, used_attributes):
    attr_type = c.split('--')[0]
    to_check = [ua for ua in used_attributes if attr_type in ua]
    attr_label = get_base_attributes(bird)

    for tc in to_check:
        if attr_label[used_attributes.index(tc)]:
            return tc
    return None

In [8]:
# this lists avoids needing CUB locally
FOLDERS = ['152.Blue_headed_Vireo', '114.Black_throated_Sparrow', '150.Sage_Thrasher', 
           '123.Henslow_Sparrow', '128.Seaside_Sparrow', '039.Least_Flycatcher', '116.Chipping_Sparrow', 
           '192.Downy_Woodpecker', '059.California_Gull', '138.Tree_Swallow', '113.Baird_Sparrow',
           '112.Great_Grey_Shrike', '069.Rufous_Hummingbird', '081.Pied_Kingfisher', '156.White_eyed_Vireo',
           '094.White_breasted_Nuthatch', '132.White_crowned_Sparrow', '058.Pigeon_Guillemot', '055.Evening_Grosbeak', 
           '048.European_Goldfinch', '121.Grasshopper_Sparrow', '057.Rose_breasted_Grosbeak', '015.Lazuli_Bunting',
           '002.Laysan_Albatross', '028.Brown_Creeper', '144.Common_Tern', '053.Western_Grebe', '072.Pomarine_Jaeger', 
           '040.Olive_sided_Flycatcher', '189.Red_bellied_Woodpecker', '045.Northern_Fulmar', '134.Cape_Glossy_Starling',
           '122.Harris_Sparrow', '035.Purple_Finch', '143.Caspian_Tern', '095.Baltimore_Oriole', '032.Mangrove_Cuckoo', 
           '126.Nelson_Sharp_tailed_Sparrow', '003.Sooty_Albatross', '004.Groove_billed_Ani', '031.Black_billed_Cuckoo',
           '071.Long_tailed_Jaeger', '190.Red_cockaded_Woodpecker', '147.Least_Tern', '170.Mourning_Warbler', 
           '062.Herring_Gull', '154.Red_eyed_Vireo', '186.Cedar_Waxwing', '006.Least_Auklet', '199.Winter_Wren', 
           '104.American_Pipit', '171.Myrtle_Warbler', '194.Cactus_Wren', '076.Dark_eyed_Junco', '024.Red_faced_Cormorant',
           '022.Chuck_will_Widow', '180.Wilson_Warbler', '052.Pied_billed_Grebe', '188.Pileated_Woodpecker', 
           '131.Vesper_Sparrow', '107.Common_Raven', '030.Fish_Crow', '043.Yellow_bellied_Flycatcher', '051.Horned_Grebe', 
           '068.Ruby_throated_Hummingbird', '037.Acadian_Flycatcher', '020.Yellow_breasted_Chat', '079.Belted_Kingfisher', 
           '182.Yellow_Warbler', '090.Red_breasted_Merganser', '179.Tennessee_Warbler', '065.Slaty_backed_Gull', 
           '099.Ovenbird', '193.Bewick_Wren', '109.American_Redstart', '096.Hooded_Oriole', '118.House_Sparrow', 
           '173.Orange_crowned_Warbler', '054.Blue_Grosbeak', '130.Tree_Sparrow', '167.Hooded_Warbler', 
           '195.Carolina_Wren', '177.Prothonotary_Warbler', '046.Gadwall', '176.Prairie_Warbler', '137.Cliff_Swallow',
           '005.Crested_Auklet', '083.White_breasted_Kingfisher', '091.Mockingbird', '151.Black_capped_Vireo',
           '158.Bay_breasted_Warbler', '056.Pine_Grosbeak', '163.Cape_May_Warbler', '034.Gray_crowned_Rosy_Finch',
           '063.Ivory_Gull', '120.Fox_Sparrow', '168.Kentucky_Warbler', '198.Rock_Wren', '164.Cerulean_Warbler',
           '026.Bronzed_Cowbird', '085.Horned_Lark', '066.Western_Gull', '017.Cardinal', '172.Nashville_Warbler', 
           '047.American_Goldfinch', '184.Louisiana_Waterthrush', '200.Common_Yellowthroat', '067.Anna_Hummingbird', 
           '023.Brandt_Cormorant', '084.Red_legged_Kittiwake', '082.Ringed_Kingfisher', '080.Green_Kingfisher', 
           '161.Blue_winged_Warbler', '064.Ring_billed_Gull', '124.Le_Conte_Sparrow', '197.Marsh_Wren', '106.Horned_Puffin',
           '149.Brown_Thrasher', '013.Bobolink', '115.Brewer_Sparrow', '125.Lincoln_Sparrow', '092.Nighthawk',
           '162.Canada_Warbler', '011.Rusty_Blackbird', '185.Bohemian_Waxwing', '050.Eared_Grebe',
           '001.Black_footed_Albatross', '070.Green_Violetear', '087.Mallard', '110.Geococcyx', '027.Shiny_Cowbird',
           '007.Parakeet_Auklet', '089.Hooded_Merganser', '012.Yellow_headed_Blackbird', '008.Rhinoceros_Auklet',
           '101.White_Pelican', '127.Savannah_Sparrow', '148.Green_tailed_Towhee', '191.Red_headed_Woodpecker', 
           '021.Eastern_Towhee', '133.White_throated_Sparrow', '075.Green_Jay', '014.Indigo_Bunting', 
           '088.Western_Meadowlark', '166.Golden_winged_Warbler', '141.Artic_Tern', '174.Palm_Warbler',
           '097.Orchard_Oriole', '169.Magnolia_Warbler', '155.Warbling_Vireo', '103.Sayornis', '135.Bank_Swallow', 
           '140.Summer_Tanager', '111.Loggerhead_Shrike', '073.Blue_Jay', '009.Brewer_Blackbird', '060.Glaucous_winged_Gull',
           '178.Swainson_Warbler', '044.Frigatebird', '196.House_Wren', '036.Northern_Flicker', '142.Black_Tern',
           '016.Painted_Bunting', '042.Vermilion_Flycatcher', '160.Black_throated_Blue_Warbler', '119.Field_Sparrow', 
           '041.Scissor_tailed_Flycatcher', '159.Black_and_white_Warbler', '029.American_Crow', '074.Florida_Jay', 
           '049.Boat_tailed_Grackle', '187.American_Three_toed_Woodpecker', '157.Yellow_throated_Vireo', 
           '086.Pacific_Loon', '129.Song_Sparrow', '153.Philadelphia_Vireo', '181.Worm_eating_Warbler',
           '146.Forsters_Tern', '183.Northern_Waterthrush', '117.Clay_colored_Sparrow', '077.Tropical_Kingbird', 
           '145.Elegant_Tern', '018.Spotted_Catbird', '093.Clark_Nutcracker', '100.Brown_Pelican', '025.Pelagic_Cormorant', 
           '019.Gray_Catbird', '165.Chestnut_sided_Warbler', '105.Whip_poor_Will', '136.Barn_Swallow', '010.Red_winged_Blackbird', 
           '098.Scott_Oriole', '038.Great_Crested_Flycatcher', '078.Gray_Kingbird', '061.Heermann_Gull', '102.Western_Wood_Pewee', 
           '108.White_necked_Raven', '175.Pine_Warbler', '139.Scarlet_Tanager', '033.Yellow_billed_Cuckoo']


In [9]:
def get_base_attributes(base_class):
    for f in FOLDERS:
        f_parts = f.split('.')
        if base_class == f_parts[1]:
            base_class_idx = int(f_parts[0]) - 1

    data = pickle.load(open(LABEL_PATH, 'rb'))
    for d in data:
        if d['class_label'] == base_class_idx:
            return d['attribute_label']
    raise ValueError("Did not find the class: {}".format(base_class))

In [10]:
correct = 0
tested = 0

for i, row in tqdm(df.iterrows(), total=len(df)):
    try:
        changed_attr = folder_to_attribute_dict['_'.join(row['image_path'].split('/')[-2].split('_')[:-1])]
    except KeyError:
        continue
    tested += 1
    filtered_columns = df.filter(like=changed_attr.split('--')[0])
    columns = filtered_columns.columns
    highest = columns[0]
    for c in columns:
        if row[c] > row[highest]:
            highest = c
    if evaluate_a_compliment:
        a_compliment = get_a_old(row['image_path'].split('/')[-3], changed_attr, used_attributes)
        if a_compliment is None:
            tested -= 1
            continue
        if highest != a_compliment:
            correct += 1
    else:
        if highest == changed_attr:
            correct += 1

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 38400/38400 [22:01<00:00, 29.05it/s]


In [11]:
print(correct / tested)

0.7324907063197026
